# 多智能体架构（Multi-Agent System）
多 Agent 协作设计模式 主要有三种：
* Supervisor 主控路由模式、
* Hierarchical 阶梯层级模式
* Peer-to-Peer 协作模式

## Multi-Agent 架构与拓扑设计
在单 Agent 架构中，随着任务复杂度的提升，单个 Prompt 或 System Message 往往会因为承担过多职责（既要做规划、又要做检索、还要写代码、审查格式）而出现上下文漂移（Context Drift）和注意力下降。解决这一瓶颈的最佳方案就是“分工协作”：将复杂系统拆解为多个角色专一（Specialized）的子 Agent，并通过合理的拓扑结构进行联动。

1. 突破单 Agent 能力瓶颈：理解 Multi-Agent 在职责分离（Separation of Concerns）、独立提示词优化与模块化解耦上的核心优势。
2. 掌握三种主流 Multi-Agent 协作拓扑：
    * Supervisor 模式（主控路由）：由一个 Central Router / Supervisor 决定下一个该由哪个子 Agent 执行。
    * Hierarchical 模式（层级管辖）：主 Agent 将任务下发给小组长（Sub-Supervisor），组长再指挥具体 Worker。
    * Peer-to-Peer / Network 模式（去中心化/自由网状）：Agent 之间直接进行对话、传接球和交叉质检（Cross-Review）。

3. 基于 LangGraph 实现 Supervisor 协作网络：编写一个由 “Supervisor 主控 Agent”、“研究员 Agent（Research）” 和 “代码员 Agent（Coder）” 协同工作的代码实战。


#### Multi-Agent 三大核心拓扑
1. Supervisor 模式（中央主管调度）：
    * 运行机制：Supervisor 节点负责解析任务，并在每轮迭代后，根据所有 Worker 的反馈动态选择下一个 Worker 节点（例如：`Researcher` -> `Coder` -> `Reviewer` -> `FINISH`）。
    * 适用场景：逻辑清晰、需要中央集中调度的复杂多步流程。

2. Hierarchical 模式（多层级分级响应）：
    * 运行机制：将子图（Sub-Graph）作为主图（Parent-Graph）的一个 Node。顶层 Supervisor 协调大方向，子图内部有自己的子 Supervisor 和 Worker 组。
    * 适用场景：超大型企业级系统，如“研发部 Agent 组”与“市场部 Agent 组”联动。

3. Collaboration / Swarm 模式（去中心化协作）：
    * 运行机制：没有固定的 Supervisor，Agent 节点可以在处理完自身逻辑后，在返回结果中直接指定下一个跳转的 Agent（“手递手传球”）。
    * 适用场景：辩论赛、多方协商、对齐校验等场景。

#### Multi-Agent 状态设计（Sub-Graph State vs Global State）
在 LangGraph 中实现多 Agent 的关键在于共享状态的设计：
* 每个子 Agent 可以维护自身的私有 Message 历史，避免将冗余的思维链（CoT）填满全局上下文。
* 节点交接时，仅向全局 `AgentState` 提交经过精炼的结构化输出（Summary / Code Artifact）。


下面的完整代码展示了如何使用 LangGraph 组装一个包含 Supervisor 调度器、研究员 Agent 与 代码编写 Agent 的多智能体协作系统：

In [ ]:
from typing import Annotated, TypedDict, Sequence, Literal
import operator

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END

# --- 1. 定义全局共享状态 (State) ---
class MultiAgentState(TypedDict):
    # 所有 Agent 共享的消息列表
    messages: Annotated[Sequence[BaseMessage], operator.add]
    # 当前指派的下一个执行者节点名称
    next_node: str

# --- 2. 定义 Supervisor 主控节点 ---
def supervisor_node(state: MultiAgentState):
    """中央主管 Agent：负责解析进展并路由到指定的 Worker 节点或结束任务"""
    messages = state["messages"]
    last_msg = messages[-1]

    print("\n👔 [Supervisor] 正在评估当前工作进度...")

    # 模拟 Supervisor 的路由推理逻辑：
    # 如果还没有研究成果 -> 分派给 Researcher
    # 如果有研究成果但还没有代码 -> 分派给 Coder
    # 如果代码已编写完成 -> 任务结束 (FINISH)

    if "研究报告" not in [m.name for m in messages]:
        print("👔 [Supervisor] 决策：指派任务给 [Researcher]")
        return {"next_node": "Researcher"}
    elif "代码实现" not in [m.name for m in messages]:
        print("👔 [Supervisor] 决策：指派任务给 [Coder]")
        return {"next_node": "Coder"}
    else:
        print("👔 [Supervisor] 决策：目标已全部完成，结束流程 (FINISH)")
        return {"next_node": "FINISH"}

# --- 3. 定义专职 Worker 节点 ---

def researcher_node(state: MultiAgentState):
    """研究员 Agent：负责搜索与整理资料"""
    print("🔬 [Researcher] 收到指令，正在进行技术调查...")
    # 模拟工具调用与分析过程
    research_result = AIMessage(
        content="【研究报告】：计算斐波那契数列的最优算法是矩阵快速幂或动态规划，空间复杂度可优化至 O(1)。",
        name="研究报告"
    )
    return {"messages": [research_result]}

def coder_node(state: MultiAgentState):
    """程序员 Agent：根据研究报告撰写代码"""
    print("💻 [Coder] 收到研究报告，正在编写高效 Python 代码...")
    code_result = AIMessage(
        content="【代码实现】：\ndef fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a",
        name="代码实现"
    )
    return {"messages": [code_result]}

# --- 4. 定义条件路由边 ---
def router(state: MultiAgentState) -> str:
    """根据 Supervisor 设置的 next_node 动态跳转"""
    return state["next_node"]

# --- 5. 构建多 Agent 图逻辑 ---
workflow = StateGraph(MultiAgentState)

# 添加所有 Agent 节点
workflow.add_node("Supervisor", supervisor_node)
workflow.add_node("Researcher", researcher_node)
workflow.add_node("Coder", coder_node)

# 设置入口节点为 Supervisor
workflow.set_entry_point("Supervisor")

# Worker 执行完成后，必须强制返回给 Supervisor 再次评估
workflow.add_edge("Researcher", "Supervisor")
workflow.add_edge("Coder", "Supervisor")

# 从 Supervisor 发出的条件路由边
workflow.add_conditional_edges(
    "Supervisor",
    router,
    {
        "Researcher": "Researcher",
        "Coder": "Coder",
        "FINISH": END
    }
)

# 编译多 Agent 图
app = workflow.compile()

# --- 6. 运行多 Agent 系统测试 ---
if __name__ == "__main__":
    print("🚀 启动 Multi-Agent 协作网络...\n")

    user_request = {"messages": [HumanMessage(content="请帮我研究并用 Python 实现一个高效率计算斐波那契数列的函数。")]}

    # 运行图并打印全流程节点切换
    for output in app.stream(user_request):
        for node_name, state_update in output.items():
            print(f"✅ 节点 [{node_name}] 处理完毕。")

    print("\n🎉 Multi-Agent 任务处理完成！")

1. Context Bloat（上下文膨胀）防御工程：
    * 思考：在上面的 Supervisor 架构中，如果 `Researcher` 和 `Coder` 在各自节点内部产生了 20 多轮中间推理或工具调用日志（Tool Calls），如果全部塞进全局 `messages` 列表，会导致成本暴涨和 LLM 混乱。
    * 工程设计：如何利用 LangGraph 的子图（Sub-Graph）或状态隔离（State Mapping）机制，只将子 Agent 最终收敛后的“最终结论”暴露给 Supervisor，而将其内部的 CoT 细节屏蔽？

2. 死循环与无限博弈拦截：
    * 在 Peer-to-Peer 架构中（例如 Coder 节点写代码，Reviewer 节点打驳回，形成闭环），容易出现两个 Agent 互不服气导致的无限博弈。
    * 思考：在 LangGraph 中除了限制全局 `recursion_limit` 之外，你会如何设置一个计数器节点（Counter Node）或强行拉入 Human-in-the-Loop 来打破死循环？


#### 框架与定位对比

| 模块 / 框架         | 核心定位                | 关键技术点                                                           | 场景应用                                     |
|-----------------|---------------------|-----------------------------------------------------------------|------------------------------------------|
| LangChain 高级检索  | 通用 AI 工作流编排         | Parent Document Retriever / Self-Query / Contextual Compression | 提升高密度文档检索命中率，减少 Token 浪费                 |
| LlamaIndex 复合查询 | 专注于数据的索引与检索增强       | VectorStoreIndex / SummaryIndex / SubQuestionQueryEngine        | 跨文档对比、全局摘要生成与多源数据整合                      |
| LangGraph 状态图   | 支持循环与复杂逻辑的 Agent 框架 | StateGraph / Nodes / Edges / Conditional Edges                  | 实现具备自我纠错（Self-Correction）循环的 ReAct Agent |
| LangGraph 高级控制  | 安全、状态恢复与人机协同        | Checkpointer (thread_id) / interrupt_before / Time Travel       | 敏感操作（转账、删除、调 API）前人工审核与恢复                |
| Multi-Agent 系统  | 复杂任务分工与降本增效         | Supervisor 模式 / Sub-Graph 状态隔离                                  | 团队级 Agent 协作，避免单模型 Context 膨胀            |



#### 企业级 IT 智能运维与审批 Agent
1. 常规查询：如果用户询问运维知识或排错指南，Agent 自动调用 RAG 向量检索 回答
2. 敏感操作：如果用户要求执行敏感命令（如“重启生产服务器”或“重置用户密码”），Agent 必须触发 Human-in-the-Loop（人工审批），挂起流程并等待管理员授权（`y/n`）。
3. 状态持久化：使用 MemorySaver 保存对话 Snapshot，确保多轮交互与断点恢复。


In [ ]:
from typing import Annotated, TypedDict, Sequence, Literal
import operator

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# ==========================================
# 1. 定义 RAG 与敏感操作工具 (Tools)
# ==========================================

@tool
def query_knowledge_base(query: str) -> str:
    """RAG 工具：查询 IT 运维知识库与故障处理手册"""
    # 模拟知识库检索结果
    if "VPN" in query.upper():
        return "【知识库命中】：VPN 连不上请检查 8443 端口是否被防火墙拦截，或重新导入 client.crt 证书。"
    return "【知识库命中】：日常运维请遵循 ISO27001 安全规范，所有高危操作需人工审批。"

@tool
def restart_production_server(server_id: str) -> str:
    """高危工具：重启指定 ID 的生产环境服务器 (需人工审批)"""
    return f"SUCCESS: 生产环境服务器 [{server_id}] 已成功重启，集群服务健康度 100%。"

tools = [query_knowledge_base, restart_production_server]
tool_map = {t.name: t for t in tools}

# ==========================================
# 2. 定义全局状态 (AgentState)
# ==========================================

class ITAgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# ==========================================
# 3. 定义图节点 (Nodes)
# ==========================================

def agent_node(state: ITAgentState):
    """LLM 思考与决策节点"""
    messages = state["messages"]
    last_msg = messages[-1]

    # 若上一条是 Tool 返回的结果，说明工具执行完毕，进行最终总结
    if isinstance(last_msg, ToolMessage):
        return {"messages": [AIMessage(content=f"操作反馈已接收：{last_msg.content}")]}

    user_query = messages[0].content

    # 逻辑模拟：根据用户请求做出不同工具决策
    if "重启" in user_query or "server" in user_query.lower():
        # 产生高危工具调用指令
        ai_msg = AIMessage(
            content="",
            tool_calls=[{
                "name": "restart_production_server",
                "args": {"server_id": "srv-prod-01"},
                "id": "call_restart_001"
            }]
        )
    else:
        # 产生普通 RAG 知识库调用指令
        ai_msg = AIMessage(
            content="",
            tool_calls=[{
                "name": "query_knowledge_base",
                "args": {"query": user_query},
                "id": "call_rag_001"
            }]
        )
    return {"messages": [ai_msg]}

def safe_tool_node(state: ITAgentState):
    """普通/常规工具执行节点 (如 RAG 查询)"""
    messages = state["messages"]
    last_msg = messages[-1]

    tool_outputs = []
    for tool_call in last_msg.tool_calls:
        if tool_call["name"] == "query_knowledge_base":
            tool_obj = tool_map[tool_call["name"]]
            output = tool_obj.invoke(tool_call["args"])
            tool_outputs.append(ToolMessage(content=str(output), tool_call_id=tool_call["id"]))

    return {"messages": tool_outputs}

def sensitive_tool_node(state: ITAgentState):
    """高危工具执行节点 (需受人工干预保护)"""
    messages = state["messages"]
    last_msg = messages[-1]

    tool_outputs = []
    for tool_call in last_msg.tool_calls:
        if tool_call["name"] == "restart_production_server":
            tool_obj = tool_map[tool_call["name"]]
            output = tool_obj.invoke(tool_call["args"])
            tool_outputs.append(ToolMessage(content=str(output), tool_call_id=tool_call["id"]))

    return {"messages": tool_outputs}

# ==========================================
# 4. 条件路由分支 (Conditional Edges)
# ==========================================

def route_decision(state: ITAgentState) -> Literal["safe_tool", "sensitive_tool", "end"]:
    messages = state["messages"]
    last_msg = messages[-1]

    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        tool_name = last_msg.tool_calls[0]["name"]
        if tool_name == "restart_production_server":
            return "sensitive_tool"
        else:
            return "safe_tool"
    return "end"

# ==========================================
# 5. 状态图组装与编译
# ==========================================

workflow = StateGraph(ITAgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("safe_tool", safe_tool_node)
workflow.add_node("sensitive_tool", sensitive_tool_node)

workflow.set_entry_point("agent")

workflow.add_conditional_edges(
    "agent",
    route_decision,
    {
        "safe_tool": "safe_tool",
        "sensitive_tool": "sensitive_tool",
        "end": END
    }
)

workflow.add_edge("safe_tool", "agent")
workflow.add_edge("sensitive_tool", "agent")

# 设置持久化与人工挂起中断点
checkpointer = MemorySaver()

# ⚠️ 仅在进入 sensitive_tool 节点前强行中断，等待人工审核
app = workflow.compile(
    checkpointer=checkpointer,
    interrupt_before=["sensitive_tool"]
)

# ==========================================
# 6. 场景模拟与测试
# ==========================================

if __name__ == "__main__":
    session_config = {"configurable": {"thread_id": "it_ticket_2026_001"}}

    print("="*60)
    print("🧪 测试场景 1：普通 RAG 运维查询 (无感知自动执行)")
    print("="*60)
    rag_input = {"messages": [HumanMessage(content="公司 VPN 经常掉线怎么排查？")]}

    for event in app.stream(rag_input, session_config):
        print(f"📍 [执行节点]: {list(event.keys())[0]}")

    state_1 = app.get_state(session_config)
    print(f"💡 最终回答: {state_1.values['messages'][-1].content}\n")

    print("="*60)
    print("🧪 测试场景 2：高危生产环境重启操作 (强行触发人工审批)")
    print("="*60)
    session_config_2 = {"configurable": {"thread_id": "it_ticket_2026_002"}}
    sensitive_input = {"messages": [HumanMessage(content="生产服务器 srv-prod-01 响应异常，请帮我直接重启服务器！")]}

    # 执行流：将在 sensitive_tool 前强行挂起
    for event in app.stream(sensitive_input, session_config_2):
        print(f"📍 [执行节点]: {list(event.keys())[0]}")

    current_state = app.get_state(session_config_2)
    print("\n🚨 【安全预警】系统阻断！检测到拟执行高危操作！")
    print(f"⏸️ 下一步准备进入的节点: {current_state.next}")
    pending_action = current_state.values["messages"][-1].tool_calls[0]
    print(f"🔍 拟执行工具: {pending_action['name']} | 参数: {pending_action['args']}\n")

    # 模拟审批授权
    approval = input("👉 运维主管请审批 (输入 'y' 确认重启, 输入 'n' 拒绝): ")
    if approval.lower() == 'y':
        print("\n✅ 审批通过，恢复图运行...")
        for event in app.stream(None, session_config_2):
            print(f"📍 [恢复后节点]: {list(event.keys())[0]}")
        final_state = app.get_state(session_config_2)
        print(f"🎉 处理完成: {final_state.values['messages'][-1].content}")
    else:
        print("\n❌ 审批拒绝，高危命令已安全取消。")

1. 考核题 1（LlamaIndex 与 RAG 选型）：
    * 假设你需要为法律部门开发一个系统，既要能回答“具体合同条款中关于违约金的界定”（细节精确查找），又要能回答“这两份长达 200 页的并发收购协议存在哪些利益冲突与条款矛盾”（跨文档对比与多子问题）。
        * 请问：你会如何在 LlamaIndex 中组装 `VectorStoreIndex`、`SummaryIndex` 与 `SubQuestionQueryEngine` 来同时满足这两个需求？

2. 考核题 2（LangGraph 架构思想）：
    * 为什么传统的 LangChain `LLMChain` / `SequentialChain` 在遇到“工具调用失败需要带上 Error 信息重新让大模型反思修正”时显得力不从心？
    * LangGraph 的 `StateGraph` 是通过什么机制（节点、边与状态 Reducer）实现这一“自我循环纠错（Self-Correction）”过程的？

3. 考核题 3（企业级 Agent 安全与状态管理）：
    * 在使用 LangGraph 的 `interrupt_before` 实现 `Human-in-the-Loop` 时，如果人类审核员不仅想做“批准/拒绝”判断，还想将 Agent 提出的参数修正后再让其执行（例如：将 Agent 准备重启的服务器 ID 从 `prod-all` 修正为 `prod-01`），在工程上应该使用 LangGraph 的什么方法来实现 State Editing？